In [ ]:
!pip install datasets pandas scikit-learn matplotlib

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("Hannes111/student_feedback_sentiment")

df = dataset["train"].to_pandas()

print(df.head())
print(df.info())
print(df.columns)

README.md:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


student_feedback_sentiment.csv:   0%|          | 0.00/60.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/724 [00:00<?, ? examples/s]

                                                text            category  \
0                                 all these are good         examination   
1                                       Satisfactory  library_facilities   
2                                         very good.  library_facilities   
3  it is quite good , depends upon teacher and th...            teaching   
4                                                Yes             labwork   

   sentiment_score sentiment  
0                1  positive  
1                1  positive  
2                1  positive  
3                1  positive  
4                1  positive  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 724 entries, 0 to 723
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   text             724 non-null    object
 1   category         724 non-null    object
 2   sentiment_score  724 non-null    int64 
 3   sentiment        724 no

In [ ]:
# Check missing values
print(df.isnull().sum())

# Remove missing values
df = df.dropna()

# Remove duplicate records
df = df.drop_duplicates()

# Display dataset shape
print("Dataset Shape:", df.shape)

# Display categories
print(df["category"].value_counts())

text               0
category           0
sentiment_score    0
sentiment          0
dtype: int64
Dataset Shape: (724, 4)
category
teaching              125
course_content        124
examination           122
library_facilities    118
extracurricular       118
labwork               117
Name: count, dtype: int64


In [ ]:
# Check sentiment labels
print(df["sentiment"].value_counts())

# Check sentiment scores
print(df["sentiment_score"].describe())

# Display sample feedback
print(df[["text", "sentiment"]].head(10))

sentiment
positive    451
negative    139
neutral     134
Name: count, dtype: int64
count    724.000000
mean       0.430939
std        0.793775
min       -1.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: sentiment_score, dtype: float64
                                                text sentiment
0                                 all these are good  positive
1                                       Satisfactory  positive
2                                         very good.  positive
3  it is quite good , depends upon teacher and th...  positive
4                                                Yes  positive
5  Our university has lot of extracurricular goin...  positive
6                                          excellent  positive
7  Teaching is excellent,but lecture delivery is ...  positive
8                        I like the question pattern  positive
9                         ALL PRACTICAL WORK IS GOOD  positive


In [ ]:
import re

# Convert text into lowercase
df["clean_text"] = df["text"].astype(str).str.lower()

# Remove special characters and numbers
df["clean_text"] = df["clean_text"].apply(
    lambda x: re.sub(r"[^a-z\s]", " ", x)
)

# Remove extra spaces
df["clean_text"] = df["clean_text"].apply(
    lambda x: re.sub(r"\s+", " ", x).strip()
)

# Display cleaned text
print(df[["text", "clean_text"]].head(10))

                                                text  \
0                                 all these are good   
1                                       Satisfactory   
2                                         very good.   
3  it is quite good , depends upon teacher and th...   
4                                                Yes   
5  Our university has lot of extracurricular goin...   
6                                          excellent   
7  Teaching is excellent,but lecture delivery is ...   
8                        I like the question pattern   
9                         ALL PRACTICAL WORK IS GOOD   

                                          clean_text  
0                                 all these are good  
1                                       satisfactory  
2                                          very good  
3  it is quite good depends upon teacher and thei...  
4                                                yes  
5  our university has lot of extracurricular goin... 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Select input and output
X = df["clean_text"]
y = df["sentiment"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert text into numbers
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Predict sentiment
y_pred = model.predict(X_test_vec)

# Check accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7379310344827587
              precision    recall  f1-score   support

    negative       0.55      0.39      0.46        28
     neutral       0.77      0.37      0.50        27
    positive       0.77      0.96      0.85        90

    accuracy                           0.74       145
   macro avg       0.70      0.57      0.60       145
weighted avg       0.73      0.74      0.71       145



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Prepare data
X = df["clean_text"]
y = df["sentiment"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Text vectorization
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Test new feedback
new_feedback = ["The teaching is very good and teachers are helpful"]

feedback_vector = vectorizer.transform(new_feedback)
prediction = model.predict(feedback_vector)

print("Student Feedback:", new_feedback[0])
print("Predicted Sentiment:", prediction[0])

Student Feedback: The teaching is very good and teachers are helpful
Predicted Sentiment: positive
